# RREF v6e reduced Colab profile

Colab notebook for the reduced RREF v6e-1 MatrixFormer/Zarr profile. It runs `scripts/rref_v6e_profile.py` through the Python API with `configs/v6e1/rref_colab_reduced_profile.yaml` and writes only compact `summary.json` plus `report.md` under `/tmp/nf-v6e1/rref_reduced/report`.

Use a TPU v6e runtime. The profile asserts JAX backend `tpu` before training; if TPU is unavailable, it stops before shard generation, training, or checkpoints.


## Runtime boundary

- Runtime: Python 3.12.
- Hardware accelerator: TPU v6e.
- Reduced task: 32x32 dense RREF over `F_1009`, 2048 traces, 500 training steps, beam width 8.
- Heavy artifacts stay in `/tmp/nf-v6e1/rref_reduced`.
- Download only `/tmp/nf-v6e1/rref_reduced/report/summary.json` and `/tmp/nf-v6e1/rref_reduced/report/report.md`.
- Do not export or commit Colab PDFs, Zarr shards, checkpoints, or raw logs.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path

REPO_URL = "https://github.com/JJYYY-JJY/tpu-verifier-normalforms.git"
REPO_ROOT = Path("/content/tpu-verifier-normalforms")
CONFIG_REL = Path("configs/v6e1/rref_colab_reduced_profile.yaml")
CONFIG_PATH = REPO_ROOT / CONFIG_REL
WORK_DIR = Path("/tmp/nf-v6e1/rref_reduced/work")
REPORT_DIR = Path("/tmp/nf-v6e1/rref_reduced/report")
SUMMARY_JSON = REPORT_DIR / "summary.json"
REPORT_MD = REPORT_DIR / "report.md"


def run_cmd(command: list[str], *, cwd: Path | None = None, redact: str | None = None) -> str:
    display_command = [part.replace(redact, "***") if redact else part for part in command]
    print("$", " ".join(display_command))
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"command failed with exit code {completed.returncode}")
    return completed.stdout


def clone_repo() -> None:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    try:
        run_cmd(["git", "clone", "--branch", "main", "--depth", "1", REPO_URL, str(REPO_ROOT)])
        return
    except RuntimeError:
        token = getpass("GitHub token with repo read access; input is hidden: ")
        if not token:
            raise
        auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        run_cmd(
            [
                "git",
                "clone",
                "--branch",
                "main",
                "--depth",
                "1",
                auth_url,
                str(REPO_ROOT),
            ],
            redact=token,
        )
        run_cmd(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_ROOT)


print("Python", platform.python_version())
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("nf-agent requires Python 3.12; switch the Colab runtime before continuing")


## Checkout and install

The install cell uses the repo's Python 3.12 package contract. If a stale checkout exists in `/content/tpu-verifier-normalforms`, it is replaced with the current `main` branch. If public clone fails because the repo is private, the cell asks for a hidden GitHub token with repo read access and resets `origin` back to the non-token URL after cloning.


In [ ]:
clone_repo()
run_cmd([sys.executable, "-m", "pip", "install", "-U", "pip"])
run_cmd(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-dev.txt", "-e", "."],
    cwd=REPO_ROOT,
)
run_cmd(["nf-agent", "--help"], cwd=REPO_ROOT)


## TPU backend check

This must print `tpu`. The probe runs in a short subprocess so the notebook kernel does not claim libtpu before the measured profile run.


In [ ]:
os.environ["JAX_PLATFORMS"] = "tpu,cpu"
probe_env = os.environ.copy()
probe_env["JAX_PLATFORMS"] = "tpu,cpu"
probe_script = "\n".join(
    [
        "import json",
        "import jax",
        "payload = {",
        "    'backend': jax.default_backend(),",
        "    'local_device_count': jax.local_device_count(),",
        "    'devices': [str(device) for device in jax.devices()],",
        "}",
        "print(json.dumps(payload))",
    ]
)
probe = subprocess.run(
    [
        sys.executable,
        "-c",
        probe_script,
    ],
    text=True,
    capture_output=True,
    check=False,
    env=probe_env,
)
if probe.stderr:
    print(probe.stderr, file=sys.stderr)
if probe.returncode != 0:
    raise RuntimeError(f"TPU backend probe failed with exit code {probe.returncode}")

probe_payload = json.loads(probe.stdout)
backend = probe_payload["backend"]
print("JAX backend:", backend)
print("local_device_count:", probe_payload["local_device_count"])
print("devices:", probe_payload["devices"])
if backend != "tpu":
    raise RuntimeError(f"expected TPU backend, got {backend!r}")


## Run reduced profile

This runs the real v6e/RREF MatrixFormer path: backward Zarr shard generation, state/action Zarr shard generation, 500 training steps, verifier-beam rollout, CPU exact replay, and compact report writing. Stage progress is printed as compact JSON records.


In [ ]:
for path in (SUMMARY_JSON, REPORT_MD):
    if path.exists():
        path.unlink()
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def load_profile_runner():
    script_path = REPO_ROOT / "scripts" / "rref_v6e_profile.py"
    spec = importlib.util.spec_from_file_location("rref_v6e_profile", script_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"could not load {script_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def print_progress(stage: str, payload: dict[str, object]) -> None:
    print(json.dumps({"stage": stage, **payload}, sort_keys=True))


runner = load_profile_runner()
payload = runner.run_profile(CONFIG_PATH, WORK_DIR, REPORT_DIR, progress=print_progress)
print(json.dumps({
    "status": payload["status"],
    "profile": payload["profile"],
    "backend": payload["status_probe"]["jax"]["backend"],
    "task": payload["task"],
    "trace_count": payload["data"]["trace_count"],
    "flat_count": payload["data"]["flat_count"],
    "train_final_step": payload["train"]["final_step"],
    "train_latest_step": payload["train"]["latest_step"],
    "train_final_loss": payload["train"]["final_loss"],
    "beam_status": payload["beam"]["status"],
    "exact_replay_ok": payload["exact_cpu_verifier"]["replay_ok"],
}, indent=2, sort_keys=True))


## Inspect compact result

The JSON should have `schema_version: rref-v6e-profile-v1`, profile `colab-v6e1-rref-reduced-32x32-mod1009`, backend `tpu`, compact data counts, training metrics, verifier-beam status, CPU exact replay status, and the no-fallback statement.


In [ ]:
payload = json.loads(SUMMARY_JSON.read_text())
print(json.dumps({
    "schema_version": payload["schema_version"],
    "status": payload["status"],
    "profile": payload["profile"],
    "backend": payload["status_probe"]["jax"]["backend"],
    "checkpoint_every": payload["train"]["checkpoint_every"],
    "learning_rate": payload["train"]["learning_rate"],
    "beam_status": payload["beam"]["status"],
    "no_fallback_statement": payload["no_fallback_statement"],
}, indent=2, sort_keys=True))
print("JSON:", SUMMARY_JSON)
print("Markdown:", REPORT_MD)


## Download compact artifacts

Download only the compact report files:

- `/tmp/nf-v6e1/rref_reduced/report/summary.json`
- `/tmp/nf-v6e1/rref_reduced/report/report.md`


In [ ]:
from google.colab import files

files.download(str(SUMMARY_JSON))
files.download(str(REPORT_MD))
